# Cross-validation of the joint dance state models: refits to the training individuals
---

## Details

&ensp;AUTHOR:&emsp;   James Foster &emsp; 2026 11 03

MODIFIED:&emsp; James Foster &emsp; 2026 11 03

DESCRIPTION:&emsp; Fits every model of [HS_circGLMM_states_joint_kfold.ipynb](HS_circGLMM_states_joint_kfold.ipynb) to the training individuals of every fold, as there, and saves the population-level posterior of each fit, so that the predictive density of the left-out individuals can be calculated separately ([HS_circGLMM_states_joint_kfold_lpd.ipynb](HS_circGLMM_states_joint_kfold_lpd.ipynb)) and recalculated without refitting.
In the first k-fold run, some fits had chains that disagreed (one chain stuck with a much lower log density), so each fit uses 6 chains and longer tuning, and a fit whose chains disagree is repeated with a new seed.

EXPECTED RUN TIME:&emsp; about 30–60 min per state model and 5 min per unimodal model on the MacBook under the watchdog, i.e. about 1–1.5 days for 10 folds. Results are saved after each fit, so the run can be stopped and resumed.

INPUTS:&emsp;    `0Data/hs_dance_reorg.csv`; the model code in `HS_circGLMM_states_joint.ipynb` (cells tagged `kfold-setup`)

OUTPUTS:&emsp;  `2Results/state_models/kfold_posteriors_<fit_tag>/fold<k>_<model>_s<seed offset>.nc` (posterior of the population-level parameters); `2Results/state_models/HS_states_joint_<fit_tag>_kfold_fits.csv` (sampler diagnostics of each fit)

## Settings
As in `HS_circGLMM_states_joint_kfold.ipynb` (keep `n_folds` and `cv_seed`, so that the folds are the same), and:
- `seed_offsets`: sampler seeds `cv_seed + offset + fold`, tried in turn until the chains agree (mean log density of the chains within `max_lp_range`); the fit with the smallest range is used
- `cv_chains`, `cv_draws`, `cv_tune`: sampler settings for each fit

In [ ]:
import json
from IPython import get_ipython

# run the data preparation and model code from HS_circGLMM_states_joint.ipynb (cells tagged 'kfold-setup')
setup_notebook = 'HS_circGLMM_states_joint.ipynb' # notebook with the model code (a frozen copy keeps a resumed run consistent)
joint_nb = json.load(open(setup_notebook, encoding='utf8'))
setup_cells = [''.join(c['source']) for c in joint_nb['cells']
               if 'kfold-setup' in c.get('metadata', {}).get('tags', [])]
print(f'{len(setup_cells)} setup cells found')


def run_setup(src):
    # run a setup cell in this namespace, stopping on errors
    get_ipython().run_cell(src).raise_error()

In [ ]:
# settings cell of HS_circGLMM_states_joint.ipynb, then overrides
run_setup(setup_cells[0]) # PYTENSOR_FLAGS
run_setup(setup_cells[1]) # settings

models_to_fit = [] # build only the cross-validation models
cv_models = ['unimodal', 'states_cond_band', 'states_cond_x_band', 'states_cond_x_band_fixed']
n_folds = 10 # None: leave one individual out
cv_seed = 20261022
test_run = False
post_dir = f'2Results/state_models/kfold_posteriors_{fit_tag}'
fits_file = f'2Results/state_models/HS_states_joint_{fit_tag}_kfold_fits.csv'

cv_chains = 6
cv_draws = 1000
cv_tune = 2000
seed_offsets = [1000, 2000, 3000]
max_lp_range = 20
if test_run:
    cv_chains, cv_draws, cv_tune = 2, 100, 100
    fits_file = fits_file.replace('.csv', '_test.csv')
    post_dir += '_test'
print(f'fit_tag: {fit_tag}, models: {cv_models}, folds: {n_folds}, test run: {test_run}')

In [ ]:
import matplotlib.pyplot as plt
for src in setup_cells[2:]: # data, design matrices, model code, initial values, model specifications
    run_setup(src)
plt.close('all')
df_full, dances_full = df.copy(), dances.copy()
print(f'{df_full["Individual"].nunique()} individuals, {len(dances_full)} dances, {len(df_full)} runs, '
      f'{len(cells_tab)} cells')

# Folds
As in `HS_circGLMM_states_joint_kfold.ipynb`.

In [ ]:
ind_runs = df_full.groupby('Individual_label').size().sort_values(ascending=False)
ind_cells = df_full.groupby('Individual_label')['cell_idx'].apply(set)
cell_inds = df_full.groupby('cell_idx')['Individual_label'].apply(set)


def keeps_all_cells(fold_inds):
    return all(len(inds - set(fold_inds)) > 0 for inds in cell_inds)


if n_folds is None:
    folds = [[ind] for ind in ind_runs.index]
else:
    folds, load = [[] for _ in range(n_folds)], np.zeros(n_folds)
    for ind, n in ind_runs.items():
        ok = [k for k in np.argsort(load, kind='stable') if keeps_all_cells(folds[k] + [ind])]
        assert ok, f'{ind} cannot be added to any fold without removing a cell from its training set'
        folds[ok[0]].append(ind)
        load[ok[0]] += n
for k, f in enumerate(folds):
    assert keeps_all_cells(f), f'fold {k} ({f}): a band x condition cell is missing from the training set'
if test_run:
    folds = folds[-1:]
pd.DataFrame({'fold': range(len(folds)), 'individuals': [', '.join(f) for f in folds],
              'runs': [int(ind_runs[f].sum()) for f in folds],
              'dances': [df_full.loc[df_full['Individual_label'].isin(f), 'Dance_label'].nunique() for f in folds]})

# Refitting to the training individuals
As in `HS_circGLMM_states_joint_kfold.ipynb`.
Only the population-level variables (no dimension longer than 20) are saved: the effects of the training individuals and their dances are not needed to predict new individuals.

In [ ]:
def recode(df_sub):
    # data and dance table for a subset of individuals, coded as in HS_circGLMM_states_joint.ipynb
    d = df_sub.drop(columns=['Dance', 'Individual']).copy()
    d = d.sort_values(['Dance_label', 'waggle_run']).reset_index(drop=True)
    d['Dance'] = d['Dance_label'].astype('category').cat.codes
    d['Individual'] = d['Individual_label'].astype('category').cat.codes
    tab = (d.groupby('Dance')
             .agg(Dance_label=('Dance_label', 'first'), Individual=('Individual', 'first'),
                  Individual_label=('Individual_label', 'first'), condition=('condition', 'first'),
                  band=('band', 'first'), cell_idx=('cell_idx', 'first'),
                  n_runs=('angle', 'size'), sun_el=('sun_el', 'mean'))
             .reset_index())
    tab = tab.join(d.groupby('Dance')['angle'].apply(dance_moments).unstack(), on='Dance')
    return d, tab


def build_cv_model(key, df_tr, dances_tr):
    global df, dances, n_dance
    df, dances, n_dance = df_tr, dances_tr, len(dances_tr)
    return build_joint_model(df, dances, model_specs[key])


def sample_cv(model, key, seed):
    start = {'nuts_sampler_kwargs': {'init_mean': init_mean(model, model_specs[key])}} if data_inits else {}
    with model:
        return pm.sample(draws=cv_draws, tune=cv_tune, chains=cv_chains, nuts_sampler=sampler,
                         target_accept=target_accept, random_seed=seed, progressbar=False, **start)


def save_posterior(idata, path):
    post = idata.posterior
    keep = [v for v in post.data_vars if all(post[v].sizes[d] <= 20 for d in post[v].dims if d not in ('chain', 'draw'))]
    post[keep].to_netcdf(path)

# Run the fits
Each fit is saved as it finishes, so an interrupted run can be resumed (saved fits are skipped).
Fits from the refit run (`kfold_refit_20261103`, same settings and first seed) are used instead of being repeated.

In [ ]:
import os, time, gc, shutil
import logging
logging.getLogger('pymc').setLevel(logging.WARNING)

os.makedirs(post_dir, exist_ok=True)
fits = pd.read_csv(fits_file) if os.path.exists(fits_file) else pd.DataFrame(columns=['fold', 'model', 'seed_offset'])
# fits of the refit run: same sampler settings and seed as the first attempt here
refit_file = f'2Results/state_models/HS_states_joint_{fit_tag}_kfold_refit.csv'
if os.path.exists(refit_file) and not test_run:
    for r in pd.read_csv(refit_file).drop_duplicates(['fold', 'model']).itertuples():
        old, new = f'{post_dir}/fold{r.fold}_{r.model}_refit.nc', f'{post_dir}/fold{r.fold}_{r.model}_s{seed_offsets[0]}.nc'
        done = ((fits['fold'] == r.fold) & (fits['model'] == r.model) & (fits['seed_offset'] == seed_offsets[0])).any()
        if os.path.exists(old) and not done:
            shutil.copy(old, new)
            fits = pd.concat([fits, pd.DataFrame([{
                'fold': r.fold, 'model': r.model, 'seed_offset': seed_offsets[0], 'chains': cv_chains,
                'tune': cv_tune, 'draws': cv_draws, 'divergences': r.divergences, 'chain_logp': r.chain_logp,
                'chain_logp_range': r.chain_logp_range, 'fit_minutes': r.fit_minutes, 'file': new}])], ignore_index=True)
    fits.to_csv(fits_file, index=False)

for k, test_inds in enumerate(folds):
    df_tr, dances_tr = recode(df_full[~df_full['Individual_label'].isin(test_inds)])
    for key in cv_models:
        for off in seed_offsets:
            prev = fits[(fits['fold'] == k) & (fits['model'] == key)]
            if (prev['chain_logp_range'] <= max_lp_range).any():
                break # chains agreed in a saved fit
            if (prev['seed_offset'] == off).any():
                continue
            t0 = time.time()
            model = build_cv_model(key, df_tr, dances_tr)
            idata_k = sample_cv(model, key, seed=cv_seed + off + k)
            path = f'{post_dir}/fold{k}_{key}_s{off}.nc'
            save_posterior(idata_k, path)
            st = idata_k.sample_stats
            lp = st['logp' if 'logp' in st else 'lp'].mean('draw').values
            row = {'fold': k, 'model': key, 'seed_offset': off, 'chains': cv_chains, 'tune': cv_tune, 'draws': cv_draws,
                   'divergences': int(st['diverging'].sum()), 'chain_logp': ' '.join(f'{x:.1f}' for x in lp),
                   'chain_logp_range': float(lp.max() - lp.min()), 'fit_minutes': (time.time() - t0) / 60, 'file': path}
            if 'n_steps' in st:
                row['mean_steps'] = float(st['n_steps'].mean())
            fits = pd.concat([fits, pd.DataFrame([row])], ignore_index=True)
            fits.to_csv(fits_file, index=False)
            print(f'fold {k} ({", ".join(test_inds)}), {key}, seed offset {off}: divergences {row["divergences"]}, '
                  f'chain logp {row["chain_logp"]}, fit {row["fit_minutes"]:.1f} min', flush=True)
            del model, idata_k
            gc.collect()
df, dances, n_dance = df_full, dances_full, len(dances_full) # restore the full data

# Fits used
The fit with the smallest range of chain log densities, for each fold and model.

In [ ]:
fits = pd.read_csv(fits_file)
best = fits.loc[fits.groupby(['fold', 'model'])['chain_logp_range'].idxmin()]
print(best[['fold', 'model', 'seed_offset', 'divergences', 'chain_logp_range', 'fit_minutes']].to_string(index=False))
print('\nFits whose chains still disagree:')
print(best[best['chain_logp_range'] > max_lp_range][['fold', 'model', 'chain_logp']].to_string(index=False))